# 04.02 · Prebuilt middleware tour

**Module 04 · Middleware, Guardrails & Human-in-the-Loop** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 150–180 min |
| 📋 Prerequisites | Lesson 04.01 (hooks and their order). |
| 🔑 Keys | `GROQ_API_KEY` (optional), `LANGSMITH_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Apply** the reliability middleware: model retry, model fallback, tool retry and tool error.
2. **Cap** runaway agents with model-call and tool-call limits, per run and per thread.
3. **Shape** what the model sees with the LLM tool selector, context editing and the to-do list.
4. **Measure** each middleware's effect (model calls, failed calls, tool calls, tokens) from a run trace, locally and in LangSmith.
5. **Stack** retry and fallback in the right order, and explain why.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Prebuilt middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in) | **Main page.** Every option used here: [tool error](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-error), [tool retry](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-retry), [model retry](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-retry), [model fallback](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-fallback), [call limits](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-call-limit), [LLM tool selector](https://docs.langchain.com/oss/python/langchain/middleware/built-in#llm-tool-selector), [context editing](https://docs.langchain.com/oss/python/langchain/middleware/built-in#context-editing), [to-do list](https://docs.langchain.com/oss/python/langchain/middleware/built-in#to-do-list) |
| [Middleware integrations](https://docs.langchain.com/oss/python/integrations/middleware/index) | Provider-specific middleware (Anthropic, OpenAI, AWS…) |
| [Models → model exceptions](https://docs.langchain.com/oss/python/langchain/models#model-exceptions) | Which errors the default `retry_on` treats as retryable |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2" "langsmith>=0.14.7,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langgraph", "langsmith",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/04-middleware-guardrails-hitl/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence one known, harmless warning (same as Lesson 02.04).
# langchain-core 1.6.9 and 1.6.10 serialize the injected ToolRuntime/Runtime while parsing input and Pydantic
# warns "PydanticSerializationUnexpectedValue ... field_name='context'". Context still arrives intact.
import warnings

warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
# 🧪 Step 5: Shared helpers for this lesson (offline): tools that fail on purpose, and a meter.
import time
from typing import Any

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool
from langchain_core.tracers.context import collect_runs
from mastery.testing import ScriptedChatModel


def call(name: str, args: dict, cid: str = "call_1") -> AIMessage:
    """An AI message that asks for one tool call (for scripted models)."""
    return AIMessage("", tool_calls=[{"name": name, "args": args, "id": cid}])


def fail(error: Exception):
    """A script item that raises instead of answering (a simulated outage)."""
    def _raise(messages):
        raise error
    return _raise


def measure(agent, question: str, **kwargs) -> dict[str, Any]:
    """Run the agent once and read the numbers from the local run trace (no account needed)."""
    start = time.perf_counter()
    with collect_runs() as cb:
        try:
            result = agent.invoke({"messages": [HumanMessage(question)]}, **kwargs)
            final = str(result["messages"][-1].content)
            tokens = sum((m.usage_metadata or {}).get("total_tokens", 0)
                         for m in result["messages"] if isinstance(m, AIMessage))
        except Exception as error:   # Show the failure in the table instead of stopping the notebook.
            final, tokens = f"💥 {type(error).__name__}: {error}", 0
    runs = cb.traced_runs
    return {
        "model calls": sum(r.run_type == "llm" for r in runs),
        "failed calls": sum(r.run_type == "llm" and r.error is not None for r in runs),
        "tool calls": sum(r.run_type == "tool" for r in runs),
        "tokens": tokens,
        "seconds": round(time.perf_counter() - start, 2),
        "final": final[:70],
    }


RESULTS: dict[str, dict] = {}   # Filled as we go; printed as one table in Part 5.
print("helpers ready")

**You should now see** your key status and `helpers ready`.

`measure()` uses `collect_runs()`, the same local run tree you met in Lesson 01.08: every model call is an `llm` run, every tool call a `tool` run, and a failed call carries an `error`. That is exactly what you will look at in LangSmith.

---
## Part 1 · Model reliability: retry and fallback

Providers fail: rate limits (429), overloaded servers (503), timeouts. Two middleware handle it:

- **`ModelRetryMiddleware`**: call the **same** model again, with exponential backoff (`initial_delay * backoff_factor ** n`, plus ±25% jitter, capped at `max_delay`).
- **`ModelFallbackMiddleware(first, *more)`**: if the model call fails, try **another** model.

In [ ]:
# 🧪 Offline: the model fails once, then answers. Without and with ModelRetryMiddleware.
from langchain.agents.middleware import ModelRetryMiddleware


def flaky_model():
    return ScriptedChatModel(responses=[fail(ConnectionError("503 Service Unavailable")), "Here is your answer."])


RESULTS["no middleware, 1 outage"] = measure(create_agent(flaky_model(), tools=[]), "Hello?")
retry = ModelRetryMiddleware(max_retries=2, initial_delay=0.1, backoff_factor=2.0, jitter=False)
RESULTS["model retry, 1 outage"] = measure(create_agent(flaky_model(), tools=[], middleware=[retry]), "Hello?")
for name in ("no middleware, 1 outage", "model retry, 1 outage"):
    print(f"{name:<26} {RESULTS[name]}")

**You should now see** the first agent end in `💥 ConnectionError: 503 Service Unavailable` (1 model call, 1 failed) and the second answer `Here is your answer.` after **2** model calls, 1 failed, about 0.1 s later.

What if the model never recovers? By default (`on_failure="continue"`) the middleware returns an `AIMessage` describing the failure, so the run **ends gracefully**. `on_failure="error"` re-raises instead. Choose deliberately: a chat UI wants a polite message, a batch job wants an exception it can count.

In [ ]:
# 🧪 Offline: a permanent outage. on_failure="continue" (default) vs "error".
def dead_model():
    return ScriptedChatModel(responses=[fail(TimeoutError("read timeout"))] * 3)


for behavior in ("continue", "error"):
    mw = ModelRetryMiddleware(max_retries=2, initial_delay=0.05, jitter=False, on_failure=behavior)
    print(f"{behavior:<9}", measure(create_agent(dead_model(), tools=[], middleware=[mw]), "Hello?"))

**You should now see** `continue` with 3 model calls (1 + 2 retries), 3 failed, and a final message `Model call failed after 3 attempts with TimeoutError: read timeout`; then `error` with the same 3 calls but `💥 TimeoutError`.

Now **fallback**: the course's pattern is *reasoning model first, fast model as the backup* (`openai/gpt-oss-120b` → `openai/gpt-oss-20b`, the same pair Project 04 uses). The scripted models get those names so you can see which one answered.

In [ ]:
# 🧪 Offline: the primary (120b) is down; ModelFallbackMiddleware switches to the 20b.
from langchain.agents.middleware import ModelFallbackMiddleware

primary = ScriptedChatModel(responses=[fail(ConnectionError("503"))], model_name="openai/gpt-oss-120b")
backup = ScriptedChatModel(responses=["Answer from the backup model."], model_name="openai/gpt-oss-20b")
agent = create_agent(primary, tools=[], middleware=[ModelFallbackMiddleware(backup)])
RESULTS["fallback 120b → 20b"] = measure(agent, "Hello?")
print(RESULTS["fallback 120b → 20b"])
print("primary calls:", len(primary.calls), "| backup calls:", len(backup.calls))

**You should now see** 2 model calls, 1 failed, the backup's answer, and `primary calls: 1 | backup calls: 1`: the primary was asked once (and failed), the backup answered.

### What just happened? Retry **inside**, fallback **outside**

Both are `wrap_model_call` middleware, so their **order** decides the behaviour (Lesson 04.01, Part 3):

| Stack | What happens on a 503 |
|---|---|
| `[ModelFallbackMiddleware(backup), ModelRetryMiddleware()]` | Retry wraps the model: the **primary** is retried first; only if all retries fail does fallback switch, and the retry layer runs again for the backup |
| `[ModelRetryMiddleware(), ModelFallbackMiddleware(backup)]` | Fallback switches **immediately**; retry only kicks in when primary *and* backup failed |

The first is usually what you want for short blips; the second for long outages (no time wasted on a dead primary). Prove it:

In [ ]:
# 🧪 Offline: the same outage, two stack orders.
def stack(order: str):
    p = ScriptedChatModel(responses=[fail(ConnectionError("503")), "primary recovered"], model_name="openai/gpt-oss-120b")
    b = ScriptedChatModel(responses=["backup answered"], model_name="openai/gpt-oss-20b")
    r = ModelRetryMiddleware(max_retries=1, initial_delay=0.05, jitter=False, on_failure="error")
    f = ModelFallbackMiddleware(b)
    return create_agent(p, tools=[], middleware=[f, r] if order == "fallback outside" else [r, f])


for order in ("fallback outside", "retry outside"):
    print(f"{order:<17}", measure(stack(order), "Hello?")["final"])

**You should now see** `fallback outside  primary recovered` (retry fixed it on the primary) and `retry outside  backup answered` (fallback switched at once).

⚠️ Note `on_failure="error"` on the **inner** retry. With the default `"continue"`, a retry that gives up returns a polite `AIMessage` instead of raising, so the outer fallback sees a *success* and never switches. Inner layers must **raise** for outer layers to react.

One more choice: **which errors deserve a retry?** The default `retry_on` retries everything it can't classify, and a provider SDK's `BadRequestError` (400) can fall in that bucket. Retrying a 400 just wastes time. Pass your own predicate:

In [ ]:
# 🧪 Offline: retry only transient failures (timeouts, connection errors, 429 and 5xx).
def transient(error: Exception) -> bool:
    status = getattr(error, "status_code", None)
    return isinstance(error, (TimeoutError, ConnectionError)) or status in (429, 500, 502, 503, 504)


class FakeBadRequest(Exception):
    status_code = 400


for error in (ConnectionError("reset"), FakeBadRequest("model_decommissioned")):
    model = ScriptedChatModel(responses=[fail(error), "ok"])
    mw = ModelRetryMiddleware(max_retries=2, initial_delay=0.05, jitter=False, retry_on=transient)
    row = measure(create_agent(model, tools=[], middleware=[mw]), "Hello?")
    print(f"{type(error).__name__:<16} model calls: {row['model calls']}  final: {row['final'][:45]}")

**You should now see** `ConnectionError  model calls: 2  final: ok` (retried) and `FakeBadRequest  model calls: 1  final: 💥 FakeBadRequest: model_decommissioned`: not retried, and raised **at once**. An error your `retry_on` rejects skips `on_failure` entirely, so an outer fallback can react to it.

In [ ]:
# 💸 Live: a primary that is guaranteed to fail (a retired model ID), falling back to the fast model.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    broken = get_model(model="groq:llama-3.3-70b-versatile")   # VERIFIED 2026-10-12: decommissioned on Groq.
    agent = create_agent(broken, tools=[], middleware=[ModelFallbackMiddleware(get_model("fast"))])
    RESULTS["live fallback (retired → fast)"] = measure(agent, "Say hello in five words.")
    print(RESULTS["live fallback (retired → fast)"])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) 2 model calls, 1 failed (the 400 `model_decommissioned`), and a short greeting from the fast model.

---
## Part 2 · Tool reliability: tool retry and tool error

In LangChain v1, **an exception raised inside a tool stops the whole run by default**. Two middleware change that:

- **`ToolRetryMiddleware`**: retry the tool with backoff (for flaky APIs). After the last failure, `on_failure="continue"` (default) returns an error `ToolMessage` to the model.
- **`ToolErrorMiddleware(on_error)`** (new in `langchain` 1.3.14): turn exceptions into error `ToolMessage`s **you** word, so the model can correct its input. Return `None` from `on_error` to let an exception propagate.

In [ ]:
# 🧪 Offline: a tool that rejects bad input. Without middleware the run crashes.
from langchain.agents.middleware import ToolErrorMiddleware


@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by its ID (format: ORD-1234)."""
    if not order_id.startswith("ORD-"):
        raise ValueError(f"bad id {order_id!r}; internal table orders_v2 at db-7.internal")   # leaks internals!
    return f"{order_id}: shipped on 2026-10-09."


def order_model():
    return ScriptedChatModel(responses=[
        call("lookup_order", {"order_id": "1234"}),                       # wrong format first
        lambda msgs: call("lookup_order", {"order_id": "ORD-1234"}, "call_2"),  # corrects itself
        lambda msgs: msgs[-1].content,                                   # answers from the tool
    ])


def on_error(error: Exception, request) -> str | None:
    if isinstance(error, ValueError):   # Name the problem, never echo the raw message (it may leak internals).
        return f"{request.tool_call['name']} failed: invalid input ({type(error).__name__}). Use the format ORD-1234."
    return None                         # Anything else: let it propagate.


RESULTS["tool raises, no middleware"] = measure(create_agent(order_model(), tools=[lookup_order]), "Where is order 1234?")
agent = create_agent(order_model(), tools=[lookup_order], middleware=[ToolErrorMiddleware(on_error)])
RESULTS["tool error middleware"] = measure(agent, "Where is order 1234?")
for name in ("tool raises, no middleware", "tool error middleware"):
    print(f"{name:<27} {RESULTS[name]}")

**You should now see** the first run end in `💥 ValueError: bad id '1234'; internal table …` after 1 model call and 1 tool call, and the second finish with `ORD-1234: shipped on 2026-10-09.` after 3 model calls and 2 tool calls: the model saw your clean error message, fixed the ID and tried again.

Now a tool that fails *transiently* (a timeout that goes away). Retry handles it; tool error is the safety net if it never recovers. The docs' recommended stack: **error outer, retry inner, retry with `on_failure="error"`** so the exception reaches the error handler after the last attempt.

In [ ]:
# 🧪 Offline: a flaky weather API (fails twice, then works), with ToolRetryMiddleware.
from langchain.agents.middleware import ToolRetryMiddleware

ATTEMPTS = {"n": 0}


@tool
def weather(city: str) -> str:
    """Current weather for a city."""
    ATTEMPTS["n"] += 1
    if ATTEMPTS["n"] <= 2:
        raise TimeoutError("weather API timed out")
    return f"{city}: 14 °C, light rain."


def weather_model():
    return ScriptedChatModel(responses=[call("weather", {"city": "Bogotá"}), lambda msgs: msgs[-1].content])


stack = [ToolErrorMiddleware(lambda e, r: f"{r.tool_call['name']} is unavailable right now ({type(e).__name__})."),
         ToolRetryMiddleware(max_retries=2, initial_delay=0.05, jitter=False, on_failure="error")]
ATTEMPTS["n"] = 0
RESULTS["tool retry (2 timeouts)"] = measure(create_agent(weather_model(), tools=[weather], middleware=stack), "Weather?")
print(RESULTS["tool retry (2 timeouts)"], "| attempts:", ATTEMPTS["n"])
ATTEMPTS["n"] = -10   # Now it fails 12 times: retries run out, the error handler answers.
RESULTS["tool retry exhausted"] = measure(create_agent(weather_model(), tools=[weather], middleware=stack), "Weather?")
print(RESULTS["tool retry exhausted"])

**You should now see** `Bogotá: 14 °C, light rain.` with `attempts: 3`, then `weather is unavailable right now (TimeoutError).` Both rows show **2 model calls but 3 tool runs**: the model asked once, and every retry is its own `tool` run in the trace. That is how you spot retries in LangSmith.

---
## Part 3 · Limits: stop runaway agents

A model that keeps calling tools (a loop) burns quota. Two limits, both counting **per run** (`run_limit`, one `invoke`) and/or **per thread** (`thread_limit`, needs a checkpointer):

| Middleware | Counts | When exceeded |
|---|---|---|
| `ModelCallLimitMiddleware` | model calls | `exit_behavior="end"` (default: stop with a message) or `"error"` |
| `ToolCallLimitMiddleware(tool_name=None)` | tool calls (all tools, or one by name) | `"continue"` (default: block the call with an error `ToolMessage`, model continues), `"error"`, or `"end"` |

In [ ]:
# 🧪 Offline: a model stuck in a loop (repeat_last=True keeps asking for the same tool).
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware


def looping_model():
    return ScriptedChatModel(responses=[call("weather", {"city": "Lima"})], repeat_last=True)


ATTEMPTS["n"] = 10   # The weather tool works now.
agent = create_agent(looping_model(), tools=[weather], middleware=[ModelCallLimitMiddleware(run_limit=4)])
RESULTS["loop + model call limit 4"] = measure(agent, "Weather?")
agent = create_agent(looping_model(), tools=[weather],
                     middleware=[ToolCallLimitMiddleware(tool_name="weather", run_limit=2, exit_behavior="end")])
RESULTS["loop + tool call limit 2 (end)"] = measure(agent, "Weather?")
for name in ("loop + model call limit 4", "loop + tool call limit 2 (end)"):
    print(f"{name:<31} {RESULTS[name]}")

**You should now see** 4 model calls and 4 tool calls ending in `Model call limits exceeded: run limit (4/4)`, and 3 model calls, 2 tool calls ending in `'weather' tool call limit reached: run limit exceeded (3/2 calls).`

Without a limit, this agent would stop only at LangGraph's `recursion_limit` (an exception after many steps), or never with a real model. **Always set a limit in production.**

---
## Part 4 · Shaping what the model sees

Three middleware change the **request** rather than handle failures.

**LLM tool selector.** With many tools, the schemas alone cost tokens and confuse the model. `LLMToolSelectorMiddleware` asks a (cheap) model first which tools matter for this query (structured output), then sends only those. `always_include` tools are always kept.

In [ ]:
# 🧪 Offline: 6 tools; the selector keeps 1 (+ search_docs, always included).
from langchain.agents.middleware import LLMToolSelectorMiddleware


def make_tool(name: str, doc: str):
    def fn(query: str) -> str:
        return f"{name}({query})"
    fn.__name__, fn.__doc__ = name, doc
    return tool(fn)


many_tools = [make_tool(n, d) for n, d in [
    ("get_weather", "Weather for a city."), ("get_stock_price", "Stock price for a ticker."),
    ("search_docs", "Search the internal docs."), ("send_invoice", "Send an invoice to a client."),
    ("book_meeting", "Book a meeting room."), ("translate", "Translate text.")]]
selector_model = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "ToolSelectionResponse", "args": {"tools": ["get_weather"]}, "id": "s1"}])])
main_model = ScriptedChatModel(responses=["(answer)"])
agent = create_agent(main_model, tools=many_tools, middleware=[
    LLMToolSelectorMiddleware(model=selector_model, max_tools=2, always_include=["search_docs"])])
agent.invoke({"messages": [HumanMessage("Will it rain in Medellín tomorrow?")]})
print("tools sent to the main model:", main_model.bound_tools)

**You should now see** `tools sent to the main model: ['get_weather', 'search_docs']`: 2 of 6 schemas. The trade-off: one extra (cheap) model call per turn, and a wrong selection means the right tool is invisible. Worth it from roughly 10+ tools.

**Context editing.** Tool outputs pile up (search results, files). `ContextEditingMiddleware` with `ClearToolUsesEdit(trigger=tokens, keep=n)` replaces **older** tool results with `[cleared]` once the conversation passes `trigger` tokens, keeping the last `keep` results.

In [ ]:
# 🧪 Offline: 5 big tool results; the 6th model call sees only the last 2 in full.
from langchain.agents.middleware import ClearToolUsesEdit, ContextEditingMiddleware


@tool
def fetch_page(url: str) -> str:
    """Fetch a web page (returns a lot of text)."""
    return f"<page {url}> " + "lorem ipsum " * 150


def what_model_sees(msgs) -> str:
    tool_results = [m for m in msgs if isinstance(m, ToolMessage)]
    cleared = sum(m.content == "[cleared]" for m in tool_results)
    return f"I see {len(tool_results)} tool results, {cleared} of them cleared."


reader = ScriptedChatModel(responses=[call("fetch_page", {"url": f"https://example.com/{i}"}, f"c{i}") for i in range(5)]
                           + [what_model_sees])
agent = create_agent(reader, tools=[fetch_page],
                     middleware=[ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=1_000, keep=2)])])
result = agent.invoke({"messages": [HumanMessage("Read five pages.")]})
print("model:", result["messages"][-1].content)
print("cleared in saved state:", sum(m.content == "[cleared]" for m in result["messages"] if isinstance(m, ToolMessage)))

**You should now see** `model: I see 5 tool results, 3 of them cleared.` and `cleared in saved state: 0`.

### What just happened?

Context editing is **transient** (Lesson 03.03's word): it edits the *request* in `wrap_model_call`, not the saved state. The full tool outputs are still in the checkpoint and in your traces; the model just doesn't pay for them. Compare with Lesson 03.03's `@before_model` + `RemoveMessage`, which deletes permanently.

**To-do list.** `TodoListMiddleware` adds a `write_todos` tool and a system-prompt section that tells the model to plan multi-step work. The plan lives in the state under `todos`, so your UI can show progress.

In [ ]:
# 🧪 Offline: the model writes a 3-step plan, then answers.
from langchain.agents.middleware import TodoListMiddleware

planner = ScriptedChatModel(responses=[
    call("write_todos", {"todos": [
        {"content": "Read the three supplier quotes", "status": "in_progress"},
        {"content": "Compare price and delivery time", "status": "pending"},
        {"content": "Recommend one supplier", "status": "pending"}]}),
    "Recommendation: Supplier B (cheapest that delivers in under a week)."])
agent = create_agent(planner, tools=[], middleware=[TodoListMiddleware()])
result = agent.invoke({"messages": [HumanMessage("Pick the best supplier from the three quotes.")]})
print("tools the model got:", planner.bound_tools)
for todo in result["todos"]:
    print(f"  [{todo['status']:<11}] {todo['content']}")
print("system prompt mentions write_todos:", "write_todos" in planner.calls[0][0].text)

**You should now see** `tools the model got: ['write_todos']`, the three to-dos with their status, and `system prompt mentions write_todos: True`.

---
## Part 5 · Measure: one table, then the same in LangSmith

Every scenario above went through `measure()`. Side by side:

In [ ]:
# 🧪 Offline: the numbers from every scenario in this notebook.
cols = ["model calls", "failed calls", "tool calls", "tokens", "seconds"]
print(f"{'scenario':<33}" + "".join(f"{c:>13}" for c in cols))
for name, row in RESULTS.items():
    print(f"{name:<33}" + "".join(f"{row[c]:>13}" for c in cols))

**You should now see** one row per scenario (9 rows offline; one more with a key). Read it like a reviewer: retry **adds** model calls but turns a crash into an answer; fallback costs one failed call; tool error turns a crash into 2 extra model calls; limits cap the loop at a known cost. Tokens are rough estimates from the scripted model.

In **LangSmith** you see the same thing per trace: every `llm` and `tool` run, which ones errored, how long each took, plus a span for each middleware hook (for example `ModelFallbackMiddleware.wrap_model_call`). The next cell sends one live fallback trace.

In [ ]:
# 💸 Live + LangSmith: one traced run with retry and fallback, in project "m04-prebuilt-middleware".
if HAVE_GROQ_KEY and is_set("LANGSMITH_API_KEY"):
    import langsmith as ls
    from langchain_core.tracers.langchain import wait_for_all_tracers
    from mastery.models import get_model

    agent = create_agent(get_model(model="groq:llama-3.3-70b-versatile"), tools=[], middleware=[
        ModelFallbackMiddleware(get_model("fast")),                                   # outer
        ModelRetryMiddleware(max_retries=1, retry_on=transient, on_failure="error"),  # inner, raises
    ])
    with ls.tracing_context(enabled=True, project_name="m04-prebuilt-middleware"):
        out = agent.invoke({"messages": [HumanMessage("Say hello in five words.")]})
    wait_for_all_tracers()
    print("Answer:", out["messages"][-1].content)
    print("Open https://smith.langchain.com → project m04-prebuilt-middleware → the newest trace.")
else:
    print("⏭️ Skipped: needs GROQ_API_KEY and LANGSMITH_API_KEY (see the setup cell).")

**You should now see** (with both keys) a greeting and the LangSmith hint. In the trace: one failed `ChatGroq` run (400 `model_decommissioned`, not retried thanks to `transient`), then a successful run on the fast model. Try it with the default `ModelRetryMiddleware(max_retries=1)` instead: the 400 is retried, and the polite `"continue"` message hides the failure from the fallback, so you get no answer at all.

---
## Part 6 · Research: what is new since this course was written?

The list of prebuilt middleware grows. This cell compares what your installed `langchain` exports with the list the course was written against (2026-10-12).

In [ ]:
# 🧪 Offline: middleware classes in your installed langchain vs. the course's list.
import langchain
import langchain.agents.middleware as mw

COURSE_LIST = {
    "ContextEditingMiddleware", "FilesystemFileSearchMiddleware", "HumanInTheLoopMiddleware",
    "LLMToolSelectorMiddleware", "ModelCallLimitMiddleware", "ModelFallbackMiddleware", "ModelRetryMiddleware",
    "PIIMiddleware", "ProviderToolSearchMiddleware", "ShellToolMiddleware", "SummarizationMiddleware",
    "TodoListMiddleware", "ToolCallLimitMiddleware", "ToolErrorMiddleware", "ToolRetryMiddleware",
}
installed = {n for n in dir(mw) if n.endswith("Middleware") and n != "AgentMiddleware"}
print("langchain", langchain.__version__, "exports", len(installed), "prebuilt middleware classes.")
print("new since the course:", sorted(installed - COURSE_LIST) or "none")
print("gone since the course:", sorted(COURSE_LIST - installed) or "none")

**You should now see** `langchain 1.4.4 exports 15 prebuilt middleware classes.` and `none` twice (with a newer `langchain`, the "new" list is your research lead). The docs page also lists middleware that live elsewhere (Deep Agents' filesystem and subagents, the LLM tool emulator, rubric grading): compare the page with this list too.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| A 400/401 is retried (wasted seconds) | The default `retry_on` retries exceptions it can't classify, which can include provider SDK errors | Pass `retry_on=transient` (Part 1): only timeouts, connection errors, 429 and 5xx |
| Fallback never triggers although the primary fails | The inner retry uses `on_failure="continue"` and returns a message instead of raising | Inner retry: `on_failure="error"` |
| Notebook pauses for many seconds | Default backoff: 1 s, 2 s, 4 s… up to 60 s | Small `initial_delay` in experiments; keep real backoff in production |
| `ToolErrorMiddleware` not catching | Retry outer with `on_failure="continue"` already converted the error | Put error **outer**, retry **inner** with `on_failure="error"` |
| `NotImplementedError` with `exit_behavior="end"` | `"end"` only works for one tool (`tool_name=`) with no other tool calls pending | Use `"continue"` (default) for global limits; `"error"` raises `ToolCallLimitExceededError` |
| `thread_limit` never resets | It counts across the whole thread | Expected; use `run_limit` for per-message caps; new `thread_id` for a new conversation |
| Tool selector picks the wrong tool | Vague tool names/docstrings, or a too-small selector model | Better docstrings (Lesson 02.01); `always_include` critical tools |
| `ImportError: cannot import name 'ToolErrorMiddleware'` | `langchain` older than 1.3.14 | Re-run Step 0 (`%pip install`), restart the kernel |

## ⚠️ Production notes

> ⚠️ **Production note: retries multiply cost and latency; budget for them.**
> 3 attempts with backoff can turn a 2 s call into 10 s and triple tokens on partial failures. Set `max_retries`, `max_delay` and a run timeout together, and alert on the *retry rate*, not just on errors.

> ⚠️ **Production note: a fallback model is a different product.**
> The 20b may follow your prompt or structured-output schema less reliably than the 120b. Run your evals (Module 10) on **both** models, and log which one answered (`response_metadata`/traces) so quality issues can be traced back.

> ⚠️ **Production note: limits are a safety net, not a design.**
> Hitting a limit means the agent was lost. Log every limit hit with the thread id, and review those traces weekly.

## 🏋️ Exercises

1. **(Easy)** Change `ModelRetryMiddleware` in Part 1 to `max_retries=0`. What do the numbers in the table become?
2. **(Medium)** Add a global `ToolCallLimitMiddleware(run_limit=1)` (default `exit_behavior="continue"`) to the looping agent. What does the model see after the first tool call? Use `format_transcript` from `mastery.agents`.
3. **(Medium)** Use `ToolRetryMiddleware(retry_on=(TimeoutError,))` and a tool that raises `PermissionError`. Is it retried?
4. **(Stretch)** Give `ModelCallLimitMiddleware(thread_limit=3)` a checkpointer (`InMemorySaver`) and call the same thread 3 times with a one-call agent. What happens on the 4th call?

<details>
<summary><b>Show solutions</b></summary>

**1.** No retry: 1 model call, 1 failed, and (with the default `on_failure="continue"`) the message `Model call failed after 1 attempts…`.

**2.** The second tool call gets a `ToolMessage` *"Tool call limit exceeded. Do not make additional tool calls."*; the scripted model keeps asking, so add a model-call limit too (a real model usually stops).

**3.** No: `PermissionError` is not in `retry_on`, so it fails at once (and without `ToolErrorMiddleware` the run stops).

**4.** The 4th `invoke` ends immediately with `Model call limits exceeded: thread limit (3/3)`: the count lives in the thread's state.
</details>

## ✅ Checkpoint

- [ ] I can add retry and fallback, and choose their order on purpose.
- [ ] I can make tool failures recoverable (tool error) or retried (tool retry), and stack both correctly.
- [ ] I can cap model and tool calls per run and per thread.
- [ ] I can explain what the tool selector, context editing and the to-do list change in the request.
- [ ] I can measure model calls, failures, tool calls and tokens from a run trace, locally and in LangSmith.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open [Prebuilt middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in) and compare its table with Part 6's output. Pick one middleware that is new to you (or that did not exist on 2026-10-12), read its options, and log in [`docs/research-log.md`](../../docs/research-log.md): what it does, which hook it uses, and one case where you would use it.

## Summary

- Reliability: `ModelRetryMiddleware`, `ModelFallbackMiddleware`, `ToolRetryMiddleware`, `ToolErrorMiddleware`. Order matters: retry inside fallback; tool error outside tool retry.
- Safety nets: `ModelCallLimitMiddleware` and `ToolCallLimitMiddleware`, per run or per thread.
- Request shaping: `LLMToolSelectorMiddleware`, `ContextEditingMiddleware` (transient), `TodoListMiddleware`.
- Measure every change: model calls, failed calls, tool calls, tokens, time; locally with `collect_runs`, in production with traces.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 04.03 · Writing custom middleware** → [`04-03-custom-middleware.ipynb`](04-03-custom-middleware.ipynb).

[Module 04 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)